# Titanic Full EDA + Feature Engineering Portfolio Notebook

**Goal:** Create a clean, professional end-to-end data science notebook using the Titanic dataset.

This notebook covers:

1. Google Colab setup  
2. Data loading  
3. Data understanding  
4. Exploratory Data Analysis  
5. Missing-value treatment  
6. Feature engineering  
7. Encoding and scaling  
8. Model-ready dataset preparation  
9. Vectors, matrices, dot product intuition  
10. PCA for practical dimensionality reduction  
11. GitHub push checklist  

> Portfolio note: This notebook is written so that another reader can understand the workflow without needing extra explanation.

## 1. Google Colab Setup

Run this section first in Google Colab.

GPU is **not required** for this task, but Colab gives free compute and makes sharing easier.

In [ ]:
# Check Python version
import sys
print("Python version:", sys.version)

# Install extra libraries if needed
# Uncomment in Colab if a package is missing
# !pip install pandas numpy matplotlib seaborn scikit-learn missingno

In [ ]:
# Core imports
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

## 2. Load the Dataset

This notebook uses the Titanic dataset from Seaborn.  
For Kaggle submission work, replace this with Kaggle's `train.csv`.

In [ ]:
# Load Titanic dataset
df = sns.load_dataset("titanic")

# Keep a backup copy
raw_df = df.copy()

df.head()

In [ ]:
print("Shape of dataset:", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

## 3. Initial Data Quality Check

We check:

- duplicate rows
- missing values
- data types
- target distribution

In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Missing values table
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing

In [ ]:
# Target distribution
df["survived"].value_counts(normalize=True).mul(100).round(2)

## 4. Exploratory Data Analysis

The target variable is `survived`.

We will analyze survival patterns across:

- gender
- passenger class
- age
- fare
- embarkation port
- family size

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="survived")
plt.title("Survival Count")
plt.xlabel("Survived: 0 = No, 1 = Yes")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="sex", hue="survived")
plt.title("Survival by Gender")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="pclass", hue="survived")
plt.title("Survival by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=df, x="age", hue="survived", bins=30, kde=True)
plt.title("Age Distribution by Survival")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=df, x="fare", hue="survived", bins=40, kde=True)
plt.title("Fare Distribution by Survival")
plt.xlabel("Fare")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="embarked", hue="survived")
plt.title("Survival by Embarkation Port")
plt.xlabel("Embarked")
plt.ylabel("Count")
plt.show()

## 5. Feature Engineering

Raw columns are not always directly useful.  
We create new features that capture stronger signals.

New features:

- `family_size = sibsp + parch + 1`
- `is_alone`
- `age_group`
- `fare_group`
- `title` extracted from passenger category where available

In [ ]:
df_fe = df.copy()

# Family features
df_fe["family_size"] = df_fe["sibsp"] + df_fe["parch"] + 1
df_fe["is_alone"] = (df_fe["family_size"] == 1).astype(int)

# Age groups
df_fe["age_group"] = pd.cut(
    df_fe["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

# Fare groups
df_fe["fare_group"] = pd.qcut(
    df_fe["fare"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

df_fe[["sibsp", "parch", "family_size", "is_alone", "age", "age_group", "fare", "fare_group"]].head()

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df_fe, x="family_size", hue="survived")
plt.title("Survival by Family Size")
plt.xlabel("Family Size")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df_fe, x="age_group", hue="survived")
plt.title("Survival by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.show()

## 6. Correlation Analysis

Correlation helps identify numerical relationships.

Note: correlation does not prove causation, but it helps detect useful patterns.

In [ ]:
numeric_df = df_fe.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 7. Prepare Model-Ready Data

We will build a clean dataset with:

- target column: `survived`
- selected useful features
- numerical preprocessing
- categorical preprocessing

Columns like `alive`, `adult_male`, `deck`, and `embark_town` are removed to avoid redundancy or high missingness.

In [ ]:
# Select useful columns
selected_features = [
    "pclass", "sex", "age", "sibsp", "parch", "fare",
    "embarked", "class", "who", "alone",
    "family_size", "is_alone", "age_group", "fare_group"
]

target = "survived"

model_df = df_fe[selected_features + [target]].copy()

model_df.head()

In [ ]:
X = model_df.drop(columns=[target])
y = model_df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_processed = preprocessor.fit_transform(X)

print("Processed feature matrix shape:", X_processed.shape)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 8. Final Model-Ready Output

At this stage, the data is ready for Classical ML models such as:

- Logistic Regression
- KNN
- Decision Tree
- Random Forest
- Support Vector Machine
- Naive Bayes

In [ ]:
# Convert processed matrix to dense only for preview/export.
# For larger datasets, keep sparse format.
feature_names = preprocessor.get_feature_names_out()
X_processed_df = pd.DataFrame(X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
                              columns=feature_names)

model_ready_df = pd.concat([X_processed_df, y.reset_index(drop=True)], axis=1)

model_ready_df.head()

In [ ]:
# Optional: save cleaned model-ready file
model_ready_df.to_csv("titanic_model_ready.csv", index=False)
print("Saved titanic_model_ready.csv")

# Part 2: Vectors and Matrices Intuition with NumPy

In data science:

- A **scalar** is a single number.
- A **vector** is one data point or one row.
- A **matrix** is the full dataset.
- A **dot product** can measure similarity between two vectors.

In [ ]:
# Scalar
scalar = 5

# Vector: one passenger-like data point
vector = np.array([3, 22, 1, 0, 7.25])

# Matrix: multiple passengers
matrix = np.array([
    [3, 22, 1, 0, 7.25],
    [1, 38, 1, 0, 71.28],
    [3, 26, 0, 0, 7.92]
])

print("Scalar:", scalar)
print("Vector shape:", vector.shape)
print("Matrix shape:", matrix.shape)

In [ ]:
# Dot product example
passenger_a = np.array([3, 22, 1, 0, 7.25])
passenger_b = np.array([1, 38, 1, 0, 71.28])

dot_product = np.dot(passenger_a, passenger_b)

print("Dot product:", dot_product)

## Dot Product Intuition

A high dot product usually means two vectors point in a similar direction or have large matching values.

In machine learning, dot products appear in:

- linear regression
- logistic regression
- neural networks
- similarity search
- recommendation systems

# Part 3: PCA — Practical Dimensionality Reduction

PCA helps reduce many features into fewer important components.

Use PCA when:

- there are many numerical features
- features are correlated
- visualization is difficult
- you want compression before modeling

Important: PCA should be applied after scaling.

In [ ]:
# PCA in three main lines
scaler = StandardScaler()
X_scaled_for_pca = scaler.fit_transform(X_processed_df)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_for_pca)

print("Original shape:", X_scaled_for_pca.shape)
print("PCA shape:", X_pca.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["survived"] = y.values

plt.figure(figsize=(7,5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="survived", alpha=0.8)
plt.title("PCA Visualization of Titanic Data")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

In [ ]:
# Explained variance visualization
pca_full = PCA()
pca_full.fit(X_scaled_for_pca)

cum_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, len(cum_variance) + 1), cum_variance, marker="o")
plt.axhline(y=0.90, linestyle="--")
plt.title("Cumulative Explained Variance by PCA Components")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.show()

## Final Summary

This notebook completed:

- Titanic full EDA
- feature engineering
- model-ready preprocessing
- vector and matrix intuition
- NumPy practice
- PCA visualization
- GitHub preparation checklist

The dataset is now ready for Classical Machine Learning models.